# Best-fit nanochat.cpp on two T4 cards (Kaggle)

Train the best-fit nanochat model with data parallel on the two T4 cards of a
Kaggle session.

One process owns each card. The two processes reduce their gradients once per
step, so the parameter update equals a one-card run. The CUDA build uses NCCL in
`backends/cuda/nccl_sync.cu`.

The best fit is 16 layers, hidden 1024, 8 heads, 4 key/value heads, and 4096
tokens. It has about 252M parameters and needs about 9.1 GiB of the 16 GiB card.
See `docs/distributed-design.md` section 10.1.

Set `CODE_REF` in the next cell to the revision to run.

## Setup

Fetch the repository, prepare the host, and build the T4 shared library. Set
`CODE_REF` to a branch or commit when the revision under test is not the default
branch.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret; the download may be rate limited")

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")

# The revision to run. `None` uses the repository default branch.
CODE_REF = None

if not REPO.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
sys.path[:0] = [str(REPO / "python"), str(REPO / "notebooks"),
                str(REPO / ".agents" / "skills" / "model-fit" / "scripts")]
print("repository:", REPO)

## NCCL

The CUDA build links NCCL. This cell finds the NCCL header and library, copies
them into a writable prefix, and sets `NANOCHAT_NCCL_PATH`. The repository rule
`third_party/nccl.bzl` reads that variable.

In [ ]:
import glob
import shutil


def _nccl_include():
    env = os.environ.get("NANOCHAT_NCCL_PATH")
    roots = ([env] if env else []) + [
        "/usr/local/cuda", "/usr", "/usr/local", "/opt/nccl",
    ]
    roots += glob.glob("/opt/conda/lib/python*/site-packages/nvidia/nccl")
    roots += glob.glob("/usr/lib/python3*/site-packages/nvidia/nccl")
    for root in roots:
        header = Path(root) / "include" / "nccl.h"
        if header.is_file():
            return header
    return None


def stage_nccl():
    header = _nccl_include()
    if header is None:
        return None
    prefix = header.parent.parent
    staging = BASE / "nccl"
    (staging / "include").mkdir(parents=True, exist_ok=True)
    (staging / "lib").mkdir(parents=True, exist_ok=True)
    shutil.copy2(header, staging / "include" / "nccl.h")
    extra = header.parent / "nccl_net.h"
    if extra.is_file():
        shutil.copy2(extra, staging / "include" / "nccl_net.h")
    libraries = sorted(glob.glob(str(prefix / "lib") + "/libnccl.so*"))
    libraries += sorted(glob.glob("/usr/lib/x86_64-linux-gnu/libnccl.so*"))
    libraries += sorted(glob.glob("/usr/lib/libnccl.so*"))
    if not libraries:
        return None
    link = staging / "lib" / "libnccl.so"
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(libraries[0])
    os.environ["NANOCHAT_NCCL_PATH"] = str(staging)
    return staging


NCCL_PREFIX = stage_nccl()
print("NCCL prefix:", NCCL_PREFIX)
assert NCCL_PREFIX is not None, "NCCL was not found; a t4 build needs it"

In [ ]:
import kaggle_setup

host = kaggle_setup.setup(url=REPO_URL, dest=REPO, ref=CODE_REF, arch="sm_75")
print("library:", host.library)
print("build key:", host.build_key)

devices = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total",
     "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(devices)
assert len(devices.splitlines()) == 2, "this notebook needs two T4 cards"

## Build the training binary

`train_main` is a separate target from the shared library. Build it for the
complete Turing config, `--config=t4` (`sm_75`, fp32, CUDA backend). Pass
`--repo_env=NANOCHAT_NCCL_PATH` so the repository rule sees the staged NCCL.

In [ ]:
build = subprocess.run(
    ["tools/nanochat", "build", "//src:train_main", "--config=t4",
     f"--repo_env=NANOCHAT_NCCL_PATH={os.environ['NANOCHAT_NCCL_PATH']}"],
    cwd=str(REPO), capture_output=True, text=True)
print(build.stdout[-2000:])
assert build.returncode == 0, build.stderr[-4000:]

TRAIN_MAIN = REPO / "bazel-bin" / "src" / "train_main"
assert TRAIN_MAIN.is_file(), TRAIN_MAIN
print("train_main:", TRAIN_MAIN)

## Data and tokenizer

One ClimbMix shard trains the model. The validation shard supplies the periodic
validation loss. Both ranks read the same files and shard documents by a stride.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub"], check=True)

from huggingface_hub import snapshot_download

DATA = BASE / "base_data_climbmix"
TRAIN_SHARD = "shard_00000.parquet"
VAL_SHARD = "shard_06542.parquet"

if not (DATA / VAL_SHARD).is_file():
    snapshot_download(repo_id="karpathy/climbmix-400b-shuffle",
                      repo_type="dataset", local_dir=str(DATA),
                      allow_patterns=[TRAIN_SHARD, VAL_SHARD],
                      token=os.environ.get("HF_TOKEN"))

print("train:", DATA / TRAIN_SHARD, "validation:", DATA / VAL_SHARD)

In [ ]:
import nanochat_cpp as nc

TOKENIZER = BASE / "tokenizer" / "tokenizer.nctoken"

if not TOKENIZER.is_file():
    TOKENIZER.parent.mkdir(parents=True, exist_ok=True)
    nc.toolchain.build(["//src/tokenizer:tok_train_main"])
    nc.toolchain.run("t0-cpu", [
        "bazel-bin/src/tokenizer/tok_train_main",
        "--parquet", f"{DATA}/{TRAIN_SHARD}",
        "--vocab-size", "32768",
        "--out", str(TOKENIZER),
    ])

tokenizer = nc.Tokenizer.load(str(TOKENIZER))
assert tokenizer.decode(tokenizer.encode("The capital of France is")) == \
    "The capital of France is", "the tokenizer round trip failed"
print("tokenizer:", tokenizer.vocab_size, "tokens")

## Best-fit run

`model_fit.BEST_FIT_T4` is the balanced pick for two T4 cards at fp32: 16
layers, hidden 1024, 8 heads, 4 key/value heads, and 4096 tokens. The `track3`
preset disables the value embeddings and sets the optimizer and the schedule.
The shape flags override the baseline shape.

`batch=1` and `grad_accum=64` give a global batch of 524288 tokens across the
two ranks. Each rank owns 512 sequences per step.

`num_iterations` bounds the run. Keep the value small for a first session.

In [ ]:
import model_fit
import parallel

RUNS = Path("/kaggle/working/runs")

best_fit = parallel.ParallelPlan(
    train_parquet=f"{DATA}/{TRAIN_SHARD}",
    val_parquet=f"{DATA}/{VAL_SHARD}",
    tokenizer=str(TOKENIZER),
    out_dir=str(RUNS / "best-fit-2xt4"),
    batch=1, grad_accum=64, num_iterations=100,
    preset="track3",
    flags=model_fit.shape_flags(model_fit.BEST_FIT_T4),
    log_every=1, eval_every=10, eval_steps=8, save_every=10, seed=42)

print(" ".join(parallel.command_for(0, best_fit)))
result = parallel.launch(best_fit, root=REPO, timeout=6 * 3600)
parallel.assert_complete(result)
print("returncodes:", result.returncodes)

## Results

Each rank logs the mean loss over its own document shard, so the two curves
differ. The reduced gradient is equal, so the two checkpoints must be
byte-identical.

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import pandas as pd

frame = pd.DataFrame(parallel.loss_rows(result))
print(frame.tail())
frame.plot(x="step", y=["rank0", "rank1"])
plt.title("best fit on two T4 cards")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()


def sha256(path):
    path = Path(path)
    return hashlib.sha256(path.read_bytes()).hexdigest() \
        if path.is_file() else "missing"


digests = {rank: sha256(best_fit.checkpoint_path(rank))
           for rank in range(best_fit.world_size)}
print(digests)
assert digests[0] == digests[1] != "missing", \
    "the two rank checkpoints differ: the gradient reduce is broken"

for rank in range(best_fit.world_size):
    path = best_fit.checkpoint_path(rank)
    size = path.stat().st_size if path.is_file() else "missing"
    print("rank", rank, path, size)

## Notes

- Set `CODE_REF` to pin the revision. Leave it `None` for the default branch.
- The two loss curves differ by design. Each rank logs the mean loss over its
  own document shard. The reduced gradient and the checkpoints are equal.
- The run directory is `/kaggle/working/runs/best-fit-2xt4`. Keep the notebook
  version to keep the outputs.
- The sidecar `<checkpoint>.cursor` holds the data cursor. A resume reads it and
  continues at the exact document.
- The CUDA build needs NCCL. Set `NANOCHAT_NCCL_PATH` when NCCL is not in a
  standard prefix.
- This notebook does not run a smoke test. The two checkpoint hashes are the
  reduce check.